# 5.3 ⚡ Performance Real: Asincronismo y Descarga Paralela (Async)

En SmartPortfolio (y en BI en general), una parte del tiempo total se va en **esperar**:

- Esperar respuestas HTTP (APIs)
- Esperar lectura de archivos remotos
- Esperar consultas a bases de datos
- Esperar servicios de terceros

Eso se llama trabajo **IO-bound** (I/O).

La idea de esta sesion:

> Si tu codigo esta esperando la red, puedes descargar en paralelo.

---

## 🎯 Objetivo

Al final podras:

1. Entender **IO-bound vs CPU-bound**
2. Usar `asyncio` para ejecutar tareas de I/O en paralelo
3. Controlar concurrencia (semaforos) para no romper rate limits
4. Implementar un mini cliente HTTP asincrono (`aiohttp`) con retries
5. Aplicarlo a SmartPortfolio: descargar precios/historicos para multiples tickers

---

## Mapa mental

```{mermaid}
flowchart LR
    A[Lista de tickers] --> B[Descarga datos]
    B --> C[Transformacion / features]
    C --> D[Analitica: retornos, volatilidad, Bollinger]
    D --> E[Optimizacion: Markowitz]
```


## 1) IO-bound vs CPU-bound

### IO-bound
Tu codigo pasa tiempo **esperando** (red, disco, DB).

- `requests.get(...)`
- `pandas.read_parquet(...)` desde S3
- `bq client.query(...)`

**Aqui** `asyncio` suele dar mejoras grandes.

### CPU-bound
Tu codigo pasa tiempo **calculando** (matrices, optimizacion, ML pesado).

- Grandes multiplicaciones de matrices
- Entrenamiento de modelos
- Optimizacion numerica intensa

Aqui normalmente se usa:

- vectorizacion (numpy)
- multiprocessing
- numba/cython
- GPU

---

## Regla practica

> Si el tiempo se va en `await` (espera), async ayuda.


## 2) Por que el codigo "sin async" es lento

Si pides 20 tickers y cada request tarda ~0.4s:

- secuencial: 20 * 0.4 = 8s
- paralelo (con limite): ~0.8s a ~2s (depende del limite)

```{mermaid}
sequenceDiagram
    participant App as App (sync)
    participant API as API
    App->>API: request #1
    API-->>App: response #1
    App->>API: request #2
    API-->>App: response #2
    App->>API: request #3
    API-->>App: response #3
```


## 3) Async: el event loop

`asyncio` permite "intercalar" tareas mientras esperan.

```{mermaid}
flowchart TD
    Loop[Event loop] --> T1[Task 1: wait HTTP]
    Loop --> T2[Task 2: wait HTTP]
    Loop --> T3[Task 3: wait HTTP]
    T1 --> Loop
    T2 --> Loop
    T3 --> Loop
```


## 4) Demo sin internet: simulando I/O con `asyncio.sleep`

Para que el notebook sea reproducible incluso sin internet,
usamos `sleep` como "latencia" de red.

La idea es medir:

- secuencial
- paralelo con `asyncio.gather`


In [1]:
from __future__ import annotations

import asyncio
import random
import time
from typing import Sequence


def fetch_sync(
    name: str,
    min_delay: float = 0.2,
    max_delay: float = 0.6,
    rng: random.Random | None = None,
) -> str:
    """Simulate a synchronous I/O call by sleeping (blocking)."""
    r = rng or random
    delay = r.uniform(min_delay, max_delay)
    time.sleep(delay)
    return f"sync:{name} ({delay:.2f}s)"


async def fetch_async(
    name: str,
    min_delay: float = 0.2,
    max_delay: float = 0.6,
    rng: random.Random | None = None,
) -> str:
    """Simulate an async I/O call by awaiting (non-blocking)."""
    r = rng or random
    delay = r.uniform(min_delay, max_delay)
    await asyncio.sleep(delay)
    return f"async:{name} ({delay:.2f}s)"


def run_sync(names: Sequence[str]) -> list[str]:
    """Run synchronous fetches sequentially."""
    start = time.perf_counter()
    out = [fetch_sync(n) for n in names]
    elapsed = time.perf_counter() - start
    print(f"[sync] elapsed: {elapsed:.2f}s")
    return out


async def run_async(names: Sequence[str]) -> list[str]:
    """Run async fetches concurrently."""
    start = time.perf_counter()
    out = await asyncio.gather(*(fetch_async(n) for n in names))
    elapsed = time.perf_counter() - start
    print(f"[async] elapsed: {elapsed:.2f}s")
    return list(out)


async def run_async_compat(names: Sequence[str]) -> list[str]:
    """
    Compatibility runner for notebooks and environments with a running event loop.

    Usage in notebooks:
        await run_async_compat(names)

    Usage in scripts:
        asyncio.run(run_async(names))
    """
    return await run_async(names)


names = [f"T{i}" for i in range(12)]

_ = run_sync(names)
_ = await run_async_compat(names)

[sync] elapsed: 5.08s
[async] elapsed: 0.54s


## 5) Controlando concurrencia: semaforos

En APIs reales hay:

- Rate limits (max requests por minuto)
- Penalizaciones (HTTP 429)
- Bloqueos temporales

Por eso NO hacemos `gather` ilimitado. Usamos un `Semaphore`.

```{mermaid}
flowchart LR
    A[gather many tasks] --> B[Semaphore limit]
    B --> C[API]
```


In [2]:
from __future__ import annotations

import asyncio
import random
import time
from typing import Sequence


async def fetch_async_limited(name: str, sem: asyncio.Semaphore) -> str:
    """Simulate async I/O with a concurrency limit."""
    async with sem:
        delay = random.uniform(0.2, 0.6)
        await asyncio.sleep(delay)
        return f"{name} ({delay:.2f}s)"


async def run_async_limited(names: Sequence[str], concurrency: int = 4) -> list[str]:
    """Run async tasks with a concurrency limit."""
    sem = asyncio.Semaphore(concurrency)

    start = time.perf_counter()

    out = await asyncio.gather(
        *(fetch_async_limited(n, sem) for n in names)
    )

    elapsed = time.perf_counter() - start
    print(f"[async limited={concurrency}] elapsed: {elapsed:.2f}s")

    return list(out)


names = [f"T{i}" for i in range(12)]

# In notebooks / Jupyter Book
_ = await run_async_limited(names, concurrency=3)
_ = await run_async_limited(names, concurrency=6)

[async limited=3] elapsed: 1.82s
[async limited=6] elapsed: 0.94s


## 6) HTTP asincrono real con `aiohttp` (para cuando haya internet)

### Instalacion

```bash
poetry add aiohttp --group dev
```

### Puntos profesionales

- `ClientSession` reutiliza conexiones (performance)
- `timeout` para no colgarte
- `retries` con backoff
- `Semaphore` para rate limits

> En este notebook el ejemplo esta listo para usar, pero requiere internet.


In [3]:
from __future__ import annotations

import asyncio
import random
from dataclasses import dataclass
from typing import Any, Dict, Optional, TYPE_CHECKING

try:
    import aiohttp
except ImportError:
    aiohttp = None  # type: ignore

if TYPE_CHECKING:  # pragma: no cover
    import aiohttp as aiohttp_types


@dataclass(frozen=True, slots=True)
class HttpClientConfig:
    timeout_s: float = 10.0
    retries: int = 3
    backoff_base_s: float = 0.5
    concurrency: int = 5


class AsyncHttpClient:
    """Minimal async HTTP client with retries and concurrency control."""

    def __init__(self, config: HttpClientConfig) -> None:
        self._cfg: HttpClientConfig = config
        self._sem = asyncio.Semaphore(config.concurrency)

    async def get_json(
        self,
        session: "aiohttp_types.ClientSession",
        url: str,
        params: Optional[Dict[str, Any]] = None,
    ) -> Dict[str, Any]:
        if aiohttp is None:
            raise RuntimeError(
                "aiohttp is not installed. Install with: poetry add aiohttp --group dev"
            )

        timeout = aiohttp.ClientTimeout(total=self._cfg.timeout_s)

        last_exc: Exception | None = None

        for attempt in range(1, self._cfg.retries + 1):
            try:
                async with self._sem:
                    async with session.get(url, params=params, timeout=timeout) as resp:
                        # Handle common rate-limit status explicitly
                        if resp.status == 429:
                            raise RuntimeError("Rate limited (HTTP 429). Try lowering concurrency.")

                        resp.raise_for_status()

                        # Some APIs return non-JSON on errors; this will raise if invalid JSON
                        return await resp.json()

            except Exception as exc:
                last_exc = exc
                if attempt >= self._cfg.retries:
                    raise

                # Exponential backoff + jitter
                backoff = self._cfg.backoff_base_s * (2 ** (attempt - 1))
                jitter = random.uniform(0, 0.2)
                await asyncio.sleep(backoff + jitter)

        # Defensive: should never reach here
        raise RuntimeError("Unexpected retry loop termination") from last_exc


EXAMPLE_URL = "https://api.github.com/users/openai"

### Ejecucion real (opcional)

In [5]:
async def demo() -> None:
    cfg = HttpClientConfig(concurrency=3, retries=3)
    client = AsyncHttpClient(cfg)

    async with aiohttp.ClientSession() as session:  # type: ignore[name-defined]
        data = await client.get_json(session, EXAMPLE_URL)
        print(data["login"], data["public_repos"])

await demo()

openai 230


## 7) Aplicacion a SmartPortfolio: descargar muchos tickers

En produccion, tu provider puede tener un metodo:

- `get_quote(ticker)`
- `get_history(ticker)`

Y luego una funcion orquestadora que descarga N tickers en paralelo con limite.

```{mermaid}
flowchart LR
    A[Tickers] --> B[Async Provider]
    B --> C[DataFrames]
    C --> D[Features / Analytics]
```


In [6]:
from __future__ import annotations

import asyncio
import hashlib
from typing import Sequence


async def fetch_quote_stub(ticker: str, sem: asyncio.Semaphore) -> float:
    """
    Stub for a quote request.
    Replace with real HTTP call in your provider.
    """
    async with sem:
        await asyncio.sleep(0.15)

        # Deterministic pseudo-price (stable across runs)
        digest = hashlib.md5(ticker.encode()).hexdigest()
        value = int(digest[:8], 16)

        return round((value % 10_000) / 100.0, 2)


async def fetch_many_quotes(
    tickers: Sequence[str],
    concurrency: int = 6,
) -> dict[str, float]:
    sem = asyncio.Semaphore(concurrency)

    results = await asyncio.gather(
        *(fetch_quote_stub(t, sem) for t in tickers)
    )

    return {t: p for t, p in zip(tickers, results)}


tickers = [
    "AAPL",
    "MSFT",
    "NVDA",
    "TSLA",
    "AMZN",
    "GOOGL",
    "META",
    "NFLX",
    "AMD",
    "INTC",
]

# Notebook / Jupyter Book
quotes = await fetch_many_quotes(tickers, concurrency=4)

quotes

{'AAPL': 1.42,
 'MSFT': 82.2,
 'NVDA': 94.95,
 'TSLA': 36.04,
 'AMZN': 96.91,
 'GOOGL': 20.79,
 'META': 99.35,
 'NFLX': 93.72,
 'AMD': 55.69,
 'INTC': 99.71}

In [7]:
import pandas as pd

df = pd.Series(quotes, name="price").sort_values()
df

AAPL      1.42
GOOGL    20.79
TSLA     36.04
AMD      55.69
MSFT     82.20
NFLX     93.72
NVDA     94.95
AMZN     96.91
META     99.35
INTC     99.71
Name: price, dtype: float64

In [8]:
import time

start = time.perf_counter()
_ = await fetch_many_quotes(tickers, concurrency=1)
print("sequential:", time.perf_counter() - start)

start = time.perf_counter()
_ = await fetch_many_quotes(tickers, concurrency=6)
print("parallel:", time.perf_counter() - start)

sequential: 1.5489631000673398
parallel: 0.31506970000918955


## 8) Checklist de buenas practicas (para 5.3)

- [ ] Identificar si el cuello de botella es IO-bound
- [ ] Usar `ClientSession` (reutilizar conexiones)
- [ ] Limitar concurrencia (`Semaphore`)
- [ ] Timeouts siempre
- [ ] Retries con backoff
- [ ] Nunca loggear secretos (keys completas)
- [ ] Diseñar providers testeables: mock/stub sin internet

---

## 🎯 Cierre

Ahora tu SmartPortfolio puede descargar datos para multiples tickers de forma eficiente.

En 5.4 vamos a usar esos datos para:

- indicadores (Bollinger, velas)
- retornos y volatilidad
- graficas de series de tiempo
